In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 04 · Variables — cada idea es una hipótesis que se mide

Este notebook **solo orquesta**. Las variables están en `src/tarifalatam/features/` (un transformador por idea, cada uno con
sus pruebas en `tests/test_features.py`), el `Pipeline` y las métricas en `src/tarifalatam/models/train.py`, y los pliegues
en `src/tarifalatam/data/folds.py`.

**Reglas escritas antes de medir** (`especificaciones/04-features-torneo-optimizacion.md`, aprobada por el grupo el
2026-10-31):

- **Mismo modelo, mismos pliegues, misma métrica.** LightGBM con parámetros fijos (los del torneo), 5 pliegues por
  anfitrión y MAE en USD por noche. Solo cambia el conjunto de variables.
- **Una variante = la referencia + una idea.** La referencia son las 13 columnas que describen el inmueble, dónde está y sus
  reglas de estadía.
- **Regla de decisión:** se conserva una idea solo si **mejora / ruido > 1** y **se conoce al momento de predecir**.
- **Columnas sospechosas** (ocupación, disponibilidad, reseñas, superanfitrión): se miden al final, solo para saber cuánto
  pesan. Esa versión no pasa al torneo ni a la app.

**Entrada:** `data/processed/train.csv.gz`. **La prueba no se abre en esta etapa.** **Salidas:** `data/processed/folds.csv`
(los pliegues que usan los notebooks 05 y 06) y `reports/04_features_tabla.csv` (una fila por variante y pliegue).

## 0. Cargar el entrenamiento

In [2]:
from tarifalatam.config import FOLDS_PATH, GROUP_COL, REPORTS_DIR, TARGET
from tarifalatam.data.load import load_train

train = load_train()  # comprueba las 65 columnas, que el id no se repita y que todos tengan anfitrión
print(f"entrenamiento: {len(train):,} anuncios x {train.shape[1]} columnas · {train[GROUP_COL].nunique():,} anfitriones")
print(train["city"].value_counts().to_string())

entrenamiento: 118,513 anuncios x 65 columnas · 53,385 anfitriones
city
rio-de-janeiro    34091
sao-paulo         33194
mexico-city       22291
santiago          14128
bogota            12205
belize             2604


## 1. Los pliegues se calculan una sola vez

`GroupKFold` por `host_id`, 5 pliegues, barajados con semilla 42: cada anfitrión queda entero en un pliegue. La asignación
se guarda por `id` de anuncio en `data/processed/folds.csv`, y los notebooks 05 y 06 la **leen** en vez de recalcularla. Así
todas las comparaciones de las tres etapas usan exactamente la misma validación.

In [3]:
from tarifalatam.data.folds import fold_indices, make_folds, save_folds

folds = make_folds(train)
save_folds(folds)
fold_pairs = fold_indices(train, folds)
for k, (fit_idx, valid_idx) in enumerate(fold_pairs):
    assert set(train.iloc[fit_idx][GROUP_COL]).isdisjoint(train.iloc[valid_idx][GROUP_COL]), "hay anfitriones en los dos lados"
    print(f"pliegue {k}: {len(valid_idx):,} anuncios de validación de {train.iloc[valid_idx][GROUP_COL].nunique():,} anfitriones")
print(f"guardado: {FOLDS_PATH.name} ({len(folds):,} filas)")

pliegue 0: 23,842 anuncios de validación de 10,677 anfitriones
pliegue 1: 24,184 anuncios de validación de 10,677 anfitriones
pliegue 2: 23,358 anuncios de validación de 10,677 anfitriones


pliegue 3: 23,593 anuncios de validación de 10,677 anfitriones
pliegue 4: 23,536 anuncios de validación de 10,677 anfitriones
guardado: folds.csv (118,513 filas)


## 2. Las ideas, escritas antes de medir

| # | Idea | Cómo se calcula | Por qué podría ayudar | ¿Se conoce al predecir? |
|---|---|---|---|---|
| 1 | Distancia al centro | km (haversine) hasta la mediana de las coordenadas de la ciudad, aprendida con el entrenamiento del pliegue | El precio cambia con la cercanía a las zonas turísticas | Sí: es la dirección |
| 2 | Amenidades | Número de amenidades y 6 indicadores por nombre exacto (`Pool`, `Air conditioning`, `Free parking on premises`, `Gym`, `Elevator`, `Hot tub`) | A igual tamaño, piscina o aire acondicionado suben el precio | Sí: el dueño sabe qué tiene el inmueble |
| 3 | Razones de capacidad | Personas por habitación (0 = estudio, cuenta como 1), personas por baño y camas por persona | Más espacio por persona = más comodidad | Sí: salen del inmueble |
| 4 | Perfil del anfitrión | Cinco conteos de anuncios del anfitrión, antigüedad del anfitrión y de su cuenta, foto de perfil e identidad verificada | Los anfitriones profesionales fijan precios distintos | Sí: el anfitrión lo sabe |
| 5 | Precio típico del barrio | El barrio codificado con el promedio del objetivo (`TargetEncoder`, con validación cruzada interna, dentro del `Pipeline`) | Resume 320 barrios en un número | Sí: el barrio se conoce |

In [4]:
from tarifalatam.features.columns import IDEA_NAMES

# ¿Se conoce al momento de predecir para un inmueble nuevo? (sección 2 de la especificación de la etapa)
KNOWN_AT_PREDICTION = {1: True, 2: True, 3: True, 4: True, 5: True}

## 3. Medir cada variante con el mismo modelo y los mismos pliegues

Cada variante se ajusta 5 veces (una por pliegue) y se mide en los anfitriones que ese ajuste no vio.

In [5]:
from tarifalatam.models.train import build_pipeline, candidates, evaluate_folds, feature_frame

fixed_model, family = candidates()["lightgbm"]  # parámetros fijos: los mismos del torneo
X, y = feature_frame(train), train[TARGET].to_numpy()

variants = {"referencia": ()} | {name: (idea,) for idea, name in IDEA_NAMES.items()}
tables = []
for name, ideas in variants.items():
    start = time.perf_counter()
    table, _ = evaluate_folds(build_pipeline(fixed_model, family, ideas=ideas), X, y, fold_pairs)
    tables.append(table.assign(variant=name))
    print(f"{name:20s} MAE {table['mae'].mean():6.2f} ± {table['mae'].std():4.2f} USD · {time.perf_counter() - start:5.0f} s")

referencia           MAE  39.90 ± 1.73 USD ·     5 s


idea_1_distancia     MAE  39.88 ± 1.71 USD ·     6 s


idea_2_amenidades    MAE  38.99 ± 1.88 USD ·    29 s


idea_3_capacidad     MAE  39.90 ± 1.70 USD ·     6 s


idea_4_anfitrion     MAE  40.21 ± 1.92 USD ·     6 s


idea_5_barrio        MAE  39.74 ± 1.69 USD ·     5 s


## 4. Tabla de ideas: mejora / ruido y decisión

- **mejora** = MAE medio de la referencia − MAE medio de la variante (positivo: la variante es mejor).
- **ruido** = desviación estándar del MAE de la variante entre los 5 pliegues.
- **mejora / ruido** menor que 1 está dentro del ruido: con otra partición la mejora podría desaparecer.
- **Complemento (no decide):** en cuántos de los 5 pliegues gana la variante y la **comparación pareada**: la diferencia
  pliegue a pliegue dividida por su propia desviación. Como los dos modelos se miden en los mismos pliegues, esta versión
  descuenta lo distintos que son los pliegues entre sí. La decisión la toma la regla escrita; una excepción la decide el
  grupo, por escrito, antes del torneo.

In [6]:
from tarifalatam.models.train import compare, improvement_over_noise, keep_idea

per_fold = pd.concat(tables, ignore_index=True)
mae = per_fold.pivot(index="fold", columns="variant", values="mae")
rows = []
for name in variants:
    idea = next((i for i, n in IDEA_NAMES.items() if n == name), None)
    gain = improvement_over_noise(mae["referencia"], mae[name])
    paired = compare(best=mae[name], other=mae["referencia"])  # diferencia positiva: la variante gana
    means = per_fold[per_fold["variant"] == name][["mape", "r2_log", "bias"]].mean()
    rows.append({
        "variante": name,
        "MAE (USD)": mae[name].mean(),
        "ruido": mae[name].std(),
        "mejora": gain["improvement"],
        "mejora / ruido": gain["ratio"] if idea else np.nan,
        "pliegues que gana": f"{gain['folds_won']} de 5" if idea else "—",
        "pareada: dif. / ruido": paired["difference_over_noise"] if idea else np.nan,
        "MAPE (%)": means["mape"],
        "R² (log)": means["r2_log"],
        "sesgo (USD)": means["bias"],
        "decisión": "—" if idea is None else ("conservar" if keep_idea(gain["ratio"], KNOWN_AT_PREDICTION[idea]) else "descartar"),
    })
ideas_table = pd.DataFrame(rows).set_index("variante")
display(ideas_table.round(3))

final_ideas = tuple(i for i, n in IDEA_NAMES.items() if ideas_table.loc[n, "decisión"] == "conservar")
print("Ideas que se conservan según la regla:", final_ideas or "ninguna (el conjunto final es la referencia)")

,MAE (USD),ruido,mejora,mejora / ruido,pliegues que gana,pareada: dif. / ruido,MAPE (%),R² (log),sesgo (USD),decisión
variante,,,,,,,,,,
referencia,39.898,1.731,0.000,NaN,—,NaN,32.704,0.636,-14.028,—
idea_1_distancia,39.879,1.714,0.019,0.011,3 de 5,0.128,32.647,0.637,-13.871,descartar
idea_2_amenidades,38.987,1.878,0.910,0.485,5 de 5,4.419,31.627,0.655,-13.592,descartar
idea_3_capacidad,39.902,1.698,-0.004,-0.002,3 de 5,-0.046,32.738,0.636,-14.036,descartar
idea_4_anfitrion,40.209,1.920,-0.312,-0.162,1 de 5,-1.118,32.972,0.634,-14.239,descartar
idea_5_barrio,39.741,1.685,0.157,0.093,5 de 5,1.541,32.757,0.637,-13.624,descartar


Ideas que se conservan según la regla: ninguna (el conjunto final es la referencia)


## 5. Conjunto final y medición con las columnas sospechosas

- **final** = referencia + ideas conservadas. Se mide una vez para confirmar; no se vuelve a decidir con esta cifra.
- **final_con_sospechosas** = final + ocupación, disponibilidad, reseñas y superanfitrión (las fechas de reseñas entran como
  días hasta el corte). Solo dice **cuánto pesan**: esa versión no sigue, porque el inversionista evalúa inmuebles nuevos,
  sin reseñas ni ocupación.

In [7]:
for name, with_suspicious in (("final", False), ("final_con_sospechosas", True)):
    start = time.perf_counter()
    pipeline = build_pipeline(fixed_model, family, ideas=final_ideas, with_suspicious=with_suspicious)
    table, _ = evaluate_folds(pipeline, X, y, fold_pairs)
    tables.append(table.assign(variant=name))
    print(f"{name:22s} MAE {table['mae'].mean():6.2f} ± {table['mae'].std():4.2f} USD · {time.perf_counter() - start:5.0f} s")

per_fold = pd.concat(tables, ignore_index=True)
mae = per_fold.pivot(index="fold", columns="variant", values="mae")
final_gain = improvement_over_noise(mae["referencia"], mae["final"])
suspicious_gain = improvement_over_noise(mae["final"], mae["final_con_sospechosas"])
suspicious_paired = compare(best=mae["final_con_sospechosas"], other=mae["final"])
print(f"\nfinal frente a la referencia: mejora {final_gain['improvement']:.2f} USD · mejora / ruido {final_gain['ratio']:.2f} · gana {final_gain['folds_won']} de 5")
print(f"con sospechosas frente al final: mejora {suspicious_gain['improvement']:.2f} USD · mejora / ruido {suspicious_gain['ratio']:.2f} · "
      f"gana {suspicious_gain['folds_won']} de 5 · pareada {suspicious_paired['difference_over_noise']:.2f}")

final                  MAE  39.90 ± 1.73 USD ·     3 s


final_con_sospechosas  MAE  37.39 ± 1.61 USD ·     6 s

final frente a la referencia: mejora 0.00 USD · mejora / ruido 0.00 · gana 0 de 5
con sospechosas frente al final: mejora 2.51 USD · mejora / ruido 1.56 · gana 5 de 5 · pareada 12.61


## 6. Guardar la tabla por pliegue

In [8]:
columns = ["variant", "fold", "mae", "mape", "rmse", "bias", "r2_log", "fit_seconds"]
output = REPORTS_DIR / "04_features_tabla.csv"
per_fold[columns].to_csv(output, index=False)
print(f"guardado: {output.name} ({len(per_fold)} filas)")
display(per_fold.pivot(index="variant", columns="fold", values="mae").round(2))

guardado: 04_features_tabla.csv (40 filas)


fold,0,1,2,3,4
variant,,,,,
final,38.35,42.47,38.29,39.88,40.50
final_con_sospechosas,35.87,39.85,36.10,37.17,37.93
idea_1_distancia,38.24,42.40,38.43,39.69,40.63
idea_2_amenidades,37.48,41.75,37.03,39.09,39.60
idea_3_capacidad,38.39,42.36,38.27,39.85,40.64
idea_4_anfitrion,38.27,42.90,38.53,40.15,41.19
idea_5_barrio,38.30,42.30,38.21,39.57,40.33
referencia,38.35,42.47,38.29,39.88,40.50


## 7. Lo que se lleva el torneo

Las ideas conservadas se anotan en `config.FINAL_IDEAS` (con un comentario que cita `reports/04_features.md`), y los
notebooks 05 y 06 las leen de ahí. Esta celda comprueba que lo anotado coincide con lo que dice la regla.

In [9]:
from tarifalatam import config

noted = getattr(config, "FINAL_IDEAS", None)
print("config.FINAL_IDEAS =", noted if noted is not None else "todavía no anotado")
if noted is not None:
    assert tuple(noted) == final_ideas, "config.FINAL_IDEAS no coincide con la regla: revisa reports/04_features.md"

config.FINAL_IDEAS = ()
